In [1]:
# apptainer exec --nv --cleanenv \
#   -B /aloy/home/ddalton:/home/ddalton \
#   -B /aloy/scratch:/aloy/scratch \
#   /aloy/home/ddalton/singularity_images/scgpt.sif \
#   /home/ddalton/.local/bin/jupyter-lab \
#   --no-browser --ip=127.0.0.1 --port=8888 --ServerApp.port_retries=0


In [2]:
"""Tutorial Annotation

Structure:
    1. Specify hyper-parameter setup for integration task
    2. Load and pre-process data
    3. Load the pre-trained scGPT model
    4. Finetune scGPT with task-specific objectives
    5. Inference with fine-tuned scGPT model
    6. Save output
"""



'Tutorial Annotation\n\nStructure:\n    1. Specify hyper-parameter setup for integration task\n    2. Load and pre-process data\n    3. Load the pre-trained scGPT model\n    4. Finetune scGPT with task-specific objectives\n    5. Inference with fine-tuned scGPT model\n    6. Save output\n'

# Region 0. Imports, Variables & Functions

Global seed set to 0
2025-09-20 18:31:16,844 - Is Cuda Available True


scGPT version: /home/ddalton/git_clones/scGPT/scgpt/__init__.py
1


# region 2. Load and pre-process data

In [ ]:

# ===== Standard library =====
import argparse
import copy
import json
import logging
import os
import pickle
import shutil
import sys
import time
from collections import Counter
from datetime import datetime
from pathlib import Path
from typing import *

# ===== Third-party =====
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scvi
import seaborn as sns
import torch
import wandb
from anndata import AnnData
from scipy.sparse import issparse
from sklearn.metrics import (
    accuracy_score,
    adjusted_rand_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    normalized_mutual_info_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    KFold,
    StratifiedGroupKFold,
    StratifiedKFold,
    train_test_split,
)
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
from torchtext.vocab import Vocab
from torchtext._torchtext import Vocab as VocabPybind

# ===== Local paths (add before importing scgpt) =====
sys.path.insert(0, "/home/ddalton/git_clones/scGPT")
sys.path.insert(0, "../")
sys.path.append("/home/ddalton/projects/scGPT_playground/")

# ===== scGPT =====
import scgpt as scg
from scgpt import SubsetsBatchSampler
from scgpt.loss import (
    criterion_neg_log_bernoulli,
    masked_mse_loss,
    masked_relative_error,
)
from scgpt.model import AdversarialDiscriminator, TransformerModel
from scgpt.preprocess import Preprocessor
from scgpt.tokenizer import random_mask_value, tokenize_and_pad_batch
from scgpt.tokenizer.gene_tokenizer import GeneVocab
from scgpt.utils import category_str2int, eval_scib_metrics, set_seed

# ===== Project helpers =====
from scanpy.pp import combat
from src.training import helpers as tr_h

# ===== One-time setup =====
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(message)s")
sc.set_figure_params(figsize=(6, 6))
os.environ["KMP_WARNINGS"] = "off"
warnings.filterwarnings("ignore")

print(f"scGPT version: {scg.__file__}")
logging.info(f"Is Cuda Available {torch.cuda.is_available()}")


print(torch.cuda.device_count())  # Check how many GPUs are available
parser = argparse.ArgumentParser(description="Script for scGPT project")

# variables
manual_parameters = {
    "scgpt_pp": "norm_log1p",  # options: raw, norm_log1p, binned
}
zero_shot_data_path = ""

# functions
def test(model: nn.Module, adata: DataLoader) -> float:
    all_counts = (
        adata.layers[input_layer_key].A
        if issparse(adata.layers[input_layer_key])
        else adata.layers[input_layer_key]
    )

    celltypes_labels = adata.obs["celltype_id"].tolist()  # make sure count from 0
    celltypes_labels = np.array(celltypes_labels)

    batch_ids = adata.obs["batch_id"].tolist()
    batch_ids = np.array(batch_ids)

    if CLS_MULTILABEL:
        d_y_multilabel = u.get_multilabel_dict_from_adata(adata)
        disease_multilabels = np.array(d_y_multilabel["Y_multilabel"])

    tokenized_test = tokenize_and_pad_batch(
        all_counts,
        gene_ids,
        max_len=manual_parameters.get("max_seq_len"),
        vocab=vocab,
        pad_token=pad_token,
        pad_value=pad_value,
        append_cls=True,  # append <cls> token at the beginning
        include_zero_gene=include_zero_gene,
    )

    input_values_test = random_mask_value(
        tokenized_test["values"],
        mask_ratio=mask_ratio,
        mask_value=mask_value,
        pad_value=pad_value,
    )

    test_data_pt = {
        "gene_ids": tokenized_test["genes"],
        "values": input_values_test,
        "target_values": tokenized_test["values"],
        "batch_labels": torch.from_numpy(batch_ids).long(),
        "celltype_labels": torch.from_numpy(celltypes_labels).long(),
    }
    if CLS_MULTILABEL:
        test_data_pt["class_multilabel"] = torch.from_numpy(disease_multilabels).float()

    test_loader = DataLoader(
        dataset=SeqDataset(test_data_pt),
        batch_size=eval_batch_size,
        shuffle=False,
        drop_last=False,
        num_workers=min(len(os.sched_getaffinity(0)), eval_batch_size // 2),
        pin_memory=True,
    )

    model.eval()
    predictions = evaluate(
        model,
        loader=test_loader,
        return_raw=True,
    )

    if CLS:
        # compute accuracy, precision, recall, f1
        accuracy = accuracy_score(celltypes_labels, predictions)
        precision = precision_score(celltypes_labels, predictions, average="macro")
        recall = recall_score(celltypes_labels, predictions, average="macro")
        macro_f1 = f1_score(celltypes_labels, predictions, average="macro")

    elif CLS_MULTILABEL:
        # compute accuracy, precision, recall, f1
        predictions_bin = (predictions > 0.5).astype(int)
        print("predictions_bin", predictions_bin)
        print("disease_multilabels", disease_multilabels)
        accuracy = accuracy_score(disease_multilabels, predictions_bin)
        precision = precision_score(
            disease_multilabels, predictions_bin, average="macro"
        )
        recall = recall_score(disease_multilabels, predictions_bin, average="macro")
        macro_f1 = f1_score(disease_multilabels, predictions_bin, average="macro")

        # computer average precision & auroc
        avg_precision = average_precision_score(
            disease_multilabels, predictions, average="macro"
        )
        auroc = roc_auc_score(
            disease_multilabels, predictions, average="micro"
        )  #! CHANGED BC SOMETIMES COMPLAINS
        print(f"Average Precision: {avg_precision:.3f}, AUROC: {auroc:.3f}")

    logger.info(
        f"Accuracy: {accuracy:.3f}, Precision: {precision:.3f}, Recall: {recall:.3f}, "
        f"Macro F1: {macro_f1:.3f}"
    )

    results = {
        "test/accuracy": accuracy,
        "test/precision": precision,
        "test/recall": recall,
        "test/macro_f1": macro_f1,
    }

    return predictions, celltypes_labels, results


def prepare_data(sort_seq_batch=False) -> Tuple[Dict[str, torch.Tensor]]:
    masked_values_train = random_mask_value(
        tokenized_train["values"],
        mask_ratio=mask_ratio,
        mask_value=mask_value,
        pad_value=pad_value,
    )
    masked_values_valid = random_mask_value(
        tokenized_valid["values"],
        mask_ratio=mask_ratio,
        mask_value=mask_value,
        pad_value=pad_value,
    )
    print(
        f"random masking at epoch {epoch:3d}, ratio of masked values in train: ",
        f"{(masked_values_train == mask_value).sum() / (masked_values_train - pad_value).count_nonzero():.4f}",
    )

    input_gene_ids_train, input_gene_ids_valid = (
        tokenized_train["genes"],
        tokenized_valid["genes"],
    )
    input_values_train, input_values_valid = masked_values_train, masked_values_valid
    target_values_train, target_values_valid = (
        tokenized_train["values"],
        tokenized_valid["values"],
    )

    tensor_batch_labels_train = torch.from_numpy(train_batch_labels).long()
    tensor_batch_labels_valid = torch.from_numpy(valid_batch_labels).long()

    tensor_celltype_labels_train = torch.from_numpy(train_celltype_labels).long()
    tensor_celltype_labels_valid = torch.from_numpy(valid_celltype_labels).long()

    if CLS_MULTILABEL:
        # 👇👇👇 Add this for multilabel
        tensor_class_multilabel_train = torch.from_numpy(
            train_disease_multilabels
        ).float()
        tensor_class_multilabel_valid = torch.from_numpy(
            valid_disease_multilabels
        ).float()

    if sort_seq_batch:  # TODO: update to random pick seq source in each traning batch
        train_sort_ids = np.argsort(train_batch_labels)
        input_gene_ids_train = input_gene_ids_train[train_sort_ids]
        input_values_train = input_values_train[train_sort_ids]
        target_values_train = target_values_train[train_sort_ids]
        tensor_batch_labels_train = tensor_batch_labels_train[train_sort_ids]
        tensor_celltype_labels_train = tensor_celltype_labels_train[train_sort_ids]

        valid_sort_ids = np.argsort(valid_batch_labels)
        input_gene_ids_valid = input_gene_ids_valid[valid_sort_ids]
        input_values_valid = input_values_valid[valid_sort_ids]
        target_values_valid = target_values_valid[valid_sort_ids]
        tensor_batch_labels_valid = tensor_batch_labels_valid[valid_sort_ids]
        tensor_celltype_labels_valid = tensor_celltype_labels_valid[valid_sort_ids]

        if CLS_MULTILABEL:
            tensor_class_multilabel_train = tensor_class_multilabel_train[
                train_sort_ids
            ]
            tensor_class_multilabel_valid = tensor_class_multilabel_valid[
                valid_sort_ids
            ]

    train_data_pt = {
        "gene_ids": input_gene_ids_train,
        "values": input_values_train,
        "target_values": target_values_train,
        "batch_labels": tensor_batch_labels_train,
        "celltype_labels": tensor_celltype_labels_train,
    }
    valid_data_pt = {
        "gene_ids": input_gene_ids_valid,
        "values": input_values_valid,
        "target_values": target_values_valid,
        "batch_labels": tensor_batch_labels_valid,
        "celltype_labels": tensor_celltype_labels_valid,
    }

    if CLS_MULTILABEL:
        train_data_pt["class_multilabel"] = tensor_class_multilabel_train
        valid_data_pt["class_multilabel"] = tensor_class_multilabel_valid

    return train_data_pt, valid_data_pt


class SeqDataset(Dataset):
    def __init__(self, data: Dict[str, torch.Tensor]):
        self.data = data

    def __len__(self):
        return self.data["gene_ids"].shape[0]

    def __getitem__(self, idx):
        return {k: v[idx] for k, v in self.data.items()}


def prepare_dataloader(
    data_pt: Dict[str, torch.Tensor],
    batch_size: int,
    shuffle: bool = False,
    intra_domain_shuffle: bool = False,
    drop_last: bool = False,
    num_workers: int = 0,
) -> DataLoader:
    if num_workers == 0:
        num_workers = min(len(os.sched_getaffinity(0)), batch_size // 2)

    dataset = SeqDataset(data_pt)

    if per_seq_batch_sample:
        # find the indices of samples in each seq batch
        subsets = []
        batch_labels_array = data_pt["batch_labels"].numpy()
        for batch_label in np.unique(batch_labels_array):
            batch_indices = np.where(batch_labels_array == batch_label)[0].tolist()
            subsets.append(batch_indices)
        data_loader = DataLoader(
            dataset=dataset,
            batch_sampler=SubsetsBatchSampler(
                subsets,
                batch_size,
                intra_subset_shuffle=intra_domain_shuffle,
                inter_subset_shuffle=shuffle,
                drop_last=drop_last,
            ),
            num_workers=num_workers,
            pin_memory=True,
        )
        return data_loader

    data_loader = DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers,
        pin_memory=True,
    )
    return data_loader


# endregion





# region 2. Load and pre-process data
# imports


# variables
mask_value = -1
pad_value = -2
pad_token = "<pad>"
special_tokens = [pad_token, "<cls>", "<eoc>"]

config = {'seed': 0, 
          'dataset_name': 'test_1', 
          'do_train': True, 
          'load_model': '/home/ddalton/projects/scGPT_playground/save/scGPT_human', 
          'mask_ratio': 0.0, 
          'epochs': 10, 
          'n_bins': 51, 
          'MVC': False, 
          'ecs_thres': 0.0, 
          'dab_weight': 0.0, 
          'lr': 0.0001, 
          'batch_size': 32, 
          'layer_size': 128, 
          'nlayers': 4, 
          'nhead': 4, 
          'dropout': 0.2, 
          'schedule_ratio': 0.9, 
          'save_eval_interval': 5, 
          'fast_transformer': True, 
          'pre_norm': False, 
          'amp': True, 
          'include_zero_gene': False, 
          'freeze': False, 
          'DSBN': False}
          

# load variables
include_zero_gene = (
    config["include_zero_gene"]
)  # if True, include zero genes among hvgs in the training
input_style = "binned"  # "normed_raw", "log1p", or "binned"
output_style = "binned"  # "normed_raw", "log1p", or "binned"manual_parameters.ADV
mask_ratio = config["mask_ratio"]
batch_size = 16
eval_batch_size = 16
max_seq_len = 3501
# functions



# load adata train
adata_valid = sc.read("/home/ddalton/projects/scGPT_playground/outputs/run-25-09-19-02/adata_valid_1.h5ad")

n_input_bins = n_bins = config["n_bins"]

# load data
adata_query = sc.read("/home/ddalton/projects/scGPT_playground/data/pp_data-25-08-14-01/data.h5ad")


# mask the genes
genes_train = adata_valid.var['gene_name']
mask_genes = adata_query.var['gene_name'].isin(genes_train)
print(f"Number of genes in test set: {adata_query.shape[1]}")
adata_query = adata_query[:, mask_genes]
print(f"Filtered data to {adata_query.shape[0]} samples and {adata_query.shape[1]} genes")


# mask samples
nan_thr= 0.9

non_nan_mask = ~np.isnan(adata_query.X)  & ~(adata_query.X==0) 
non_nan_mask_pct = np.sum(non_nan_mask, axis=1) / adata_query.X.shape[1]

# mask samples that have less than 30% non-NaN values
mask_samples_nan = non_nan_mask_pct >= nan_thr 

print(f"{nan_thr} Keeping {np.sum(mask_samples_nan)} samples out of {adata_query.X.shape[0]} ({np.sum(mask_samples_nan)/adata_query.X.shape[0]*100:.2f}%)")


zero_thr = 0.5
non_zero_mask = ~(adata_query.X==0) 
non_zero_mask_pct = np.sum(non_zero_mask, axis=1) / adata_query.X.shape[1]

# mask samples that have less than 30% non-NaN values
mask_samples_zero = non_zero_mask_pct >= zero_thr 

print(f"{zero_thr} Keeping {np.sum(mask_samples_zero)} samples out of {adata_query.X.shape[0]} ({np.sum(mask_samples_zero)/adata_query.X.shape[0]*100:.2f}%)")

mask_samples_comb = mask_samples_nan & mask_samples_zero
print(f"Combined: Keeping {np.sum(mask_samples_comb)} samples out of {adata_query.X.shape[0]} ({np.sum(mask_samples_comb)/adata_query.X.shape[0]*100:.2f}%)")

# apply the mask to the AnnData object
adata_query = adata_query[mask_samples_comb, :]


# config parameters
data_is_raw = True
filter_gene_by_counts = False

if manual_parameters.get("scgpt_pp") == "norm_log1p":
    #! QUICK FIX BECAUSE SCGPT IS FUCKING USELESS AND MESSES UP NANs
    # substitue NaNs with 0s
    adata_query.X = adata_query.X.toarray() if issparse(adata_query.X) else adata_query.X

    # change

    # adata_query.X = np.nan_to_num(adata_query.X, nan=0.0)

    #! CHANGE IN FUTURE!
    #! we are introducing log2 scaled data - we should NOT apply log1 on the log2
    # adata_query.X = np.power(2, adata_query.X) - 1 # originally it was log2(X+1)



    # set up the preprocessor, use the args to config the workflow
    preprocessor = Preprocessor(
        use_key="X",  # the key in adata_query.layers to use as raw data
        filter_gene_by_counts=False,  # step 1
        filter_cell_by_counts=False,  # step 2 #! WE HAVE CASES WHERE EVERYTHING IS 0 - WE SHOULD ACTIVATE THIS!
        normalize_total=1e4,  # 3. whether to normalize the raw data and to what sum
        result_normed_key="X_normed",  # the key in adata_query.layers to store the normalized data
        log1p=True,  # 4. whether to log1p the normalized data
        result_log1p_key="X_log1p",
        subset_hvg=False,  # 5. whether to subset the raw data to highly variable genes
        hvg_flavor="seurat_v3" if True else "cell_ranger",
        binning=config.get("n_bins"),  # 6. whether to bin the raw data and to what number of bins
        result_binned_key="X_binned",  # the key in adata_query.layers to store the binned data
        )

    # define mapping of input layer
    d_input_layer = {  # the values of this map coorespond to the keys in preprocessing
                    "normed_raw": "X_normed",
                    "log1p": "X_normed",
                    "binned": "X_binned",
                    }



#! WHAT IS THIS
print("config.load_model", config.get("load_model"))
if config.get("load_model") is not None:
    model_dir = Path(config.get("load_model"))
    model_config_file = model_dir / "args.json"
    model_file = model_dir / "best_model.pt"
    vocab_file = model_dir / "vocab.json"

    vocab = GeneVocab.from_file(vocab_file)

    for s in special_tokens:
        if s not in vocab:
            vocab.append_token(s)

    adata_query.var["id_in_vocab"] = [
        1 if gene in vocab else -1 for gene in adata_query.var["gene_name"]
    ]
    gene_ids_in_vocab = np.array(adata_query.var["id_in_vocab"])
    print(
        f"match {np.sum(gene_ids_in_vocab >= 0)}/{len(gene_ids_in_vocab)} genes "
        f"in vocabulary of size {len(vocab)}."
    )
    adata_query = adata_query[:, adata_query.var["id_in_vocab"] >= 0]

    # model
    with open(model_config_file, "r") as f:
        model_configs = json.load(f)
    print(
        f"Resume model from {model_file}, the model args will override the "
        f"config {model_config_file}."
    )
    embsize = model_configs["embsize"]
    nhead = model_configs["nheads"]
    d_hid = model_configs["d_hid"]
    nlayers = model_configs["nlayers"]
    n_layers_cls = model_configs["n_layers_cls"]

# convert batch ids to integers
_batch_ids = adata_query.obs["batch_id"].tolist()
num_batch_types = adata_query.obs["batch_id"].nunique()
_remap_dict = {k: i for i, k in enumerate(sorted(set(_batch_ids)))}
adata_query.obs["batch_id"] = np.array([_remap_dict[b] for b in _batch_ids], dtype=int)  # update the batch ids in adata_query.obs

# seperate data

mask_nans = np.isnan(adata_query.X)
adata_query.X[mask_nans] = 0.0    # set to 0

preprocessor(adata_query, batch_key=None)


# set to pad value 
adata_query.X[mask_nans] = pad_value



#! ASSESS MAX VALUES AFTER PP

genes = adata_valid.var["gene_name"].tolist()


if config["load_model"] is None:
    vocab = Vocab(
        VocabPybind(genes + special_tokens, None)
    )  # bidirectional lookup [gene <-> int]
vocab.set_default_index(vocab["<pad>"])
gene_ids = np.array(vocab(genes), dtype=int)


# generate celltype label
celltype_id_labels = adata_query.obs["celltype"].astype("category").cat.codes.values
celltypes = adata_query.obs["celltype"].unique()
num_types = len(np.unique(celltype_id_labels))
id2type = dict(enumerate(adata_query.obs["celltype"].astype("category").cat.categories))
adata_query.obs["celltype_id"] = celltype_id_labels

# endregion



# region 3. Load the pre-trained scGPT model
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
# device = torch.device("cpu")

ntokens = len(vocab)  # size of vocabulary


ntokens = len(vocab)  # size of vocabulary
model = TransformerModel(
    ntokens,
    embsize,
    nhead,
    d_hid,
    nlayers,
    nlayers_cls=3,
    vocab=vocab,
    pad_token=pad_token,
    pad_value=pad_value,

)



try:
    model.load_state_dict(torch.load(model_file))
    print(f"Loading all model params from {model_file}")
except:
    # only load params that are in the model and match the size
    model_dict = model.state_dict()
    pretrained_dict = torch.load(model_file)
    pretrained_dict = {
        k: v
        for k, v in pretrained_dict.items()
        if k in model_dict and v.shape == model_dict[k].shape
    }
    for k, v in pretrained_dict.items():
        # print(f"Loading params {k} with shape {v.shape}")
        pass
    model_dict.update(pretrained_dict)
    model.load_state_dict(model_dict)


model.to(device)


input_layer_key = d_input_layer[input_style]


all_counts = (
    adata_query.layers[input_layer_key].A
    if issparse(adata_query.layers[input_layer_key])
    else adata_query.layers[input_layer_key]
)

celltypes_labels = adata_query.obs["celltype_id"].tolist()  # make sure count from 0
celltypes_labels = np.array(celltypes_labels)

batch_ids = adata_query.obs["batch_id"].tolist()
batch_ids = np.array(batch_ids)

tokenized_test = tokenize_and_pad_batch(
    all_counts,
    gene_ids,
    max_len=max_seq_len,
    vocab=vocab,
    pad_token=pad_token,
    pad_value=pad_value,
    append_cls=True,  # append <cls> token at the beginning
    include_zero_gene=include_zero_gene,
)

input_values_test = random_mask_value(
    tokenized_test["values"],
    mask_ratio=mask_ratio,
    mask_value=mask_value,
    pad_value=pad_value,
)

test_data_pt = {
    "gene_ids": tokenized_test["genes"],
    "values": input_values_test,
    "target_values": tokenized_test["values"],
    "batch_labels": torch.from_numpy(batch_ids).long(),
    "celltype_labels": torch.from_numpy(celltypes_labels).long(),
}

test_loader = DataLoader(
    dataset=SeqDataset(test_data_pt),
    batch_size=eval_batch_size//2,
    shuffle=False,
    drop_last=False,
    num_workers=min(len(os.sched_getaffinity(0)), eval_batch_size // 2),
    pin_memory=True,
)

model.eval()

predictions = list()
with torch.no_grad():
    for batch_data in tqdm(test_loader):
        input_gene_ids = batch_data["gene_ids"].to(device)
        input_values = batch_data["values"].to(device)
        target_values = batch_data["target_values"].to(device)
        batch_labels = batch_data["batch_labels"].to(device)
        celltype_labels = batch_data["celltype_labels"].to(device)

        src_key_padding_mask = input_gene_ids.eq(vocab[pad_token])
        with torch.cuda.amp.autocast(enabled=config["amp"]):
            output_dict = model(
                input_gene_ids,
                input_values,
                src_key_padding_mask=src_key_padding_mask,
                batch_labels=None,
                CLS=True, 
                CCE=False,
                MVC=False,
                ECS=False,
                # generative_training = False,
            )
            
            output_values = output_dict["cls_output"]

            preds = torch.sigmoid(output_values).cpu().numpy()
            preds_bin = (preds > 0.5).astype(int)
        predictions.append(preds)




print("predictions", predictions)

logging.info("region 5")
tr_h.log_cpu_memory_usage()
tr_h.log_gpu_memory_usage()

# endregion


Number of genes in test set: 20608
Filtered data to 35626 samples and 3501 genes


0.9 Keeping 30895 samples out of 35626 (86.72%)
0.5 Keeping 34286 samples out of 35626 (96.24%)
Combined: Keeping 30895 samples out of 35626 (86.72%)
config.load_model /home/ddalton/projects/scGPT_playground/save/scGPT_human
match 3501/3501 genes in vocabulary of size 60697.
Resume model from /home/ddalton/projects/scGPT_playground/save/scGPT_human/best_model.pt, the model args will override the config /home/ddalton/projects/scGPT_playground/save/scGPT_human/args.json.
[DEBUG] Starting preprocessing
[DEBUG] key_to_process = None
[DEBUG] adata.X type=<class 'numpy.ndarray'>, shape=(30895, 3501)
scGPT - INFO - Normalizing total counts ...
scGPT - INFO - Log1p transforming ...
scGPT - WARNING - The input data seems to be already log1p transformed. Set `log1p=False` to avoid double log1p transform.
scGPT - INFO - Binning data ...
[DEBUG] Binning data with n_bins=51
[DEBUG] layer_data type=<class 'numpy.ndarray'>, shape=(30895, 3501)


In [6]:
adata_valid.obs

,ids,dataset,dataset_id,batch,batch_id,dsaid,tissue,n_genes,disease,celltype,disease_study,library,doid_study,doid_id,do_id,doid_disease,celltype_id,sample_id,test_split_1
14516,DSA01344.GSM4595284.Case,GSE152004,GSE152004,1931,534,DSA01344,Nasal airway epithelium,19402,Asthma,DOID:2841,Asthma,RNA-Seq,D,DOID:2841,DOID:2841,asthma,61,GSM4595284,0
101584,DSA08967.TCGA-EP-A2KC-01A.Case,TCGA-LIHC,TCGA-LIHC,1991,552,DSA08967,nan,18983,Liver Hepatocellular Carcinoma,DOID:684,Liver Hepatocellular Carcinoma,RNA-Seq,D,DOID:684,DOID:684,hepatocellular carcinoma,14,TCGA-EP-A2KC-01A,0
19370,DSA01703.GSM3427890.Case,GSE121212,GSE121212,931,258,DSA01703,Skin,19402,Atopic Dermatitis,DOID:3310,Atopic Dermatitis,RNA-Seq,D,DOID:3310,DOID:3310,atopic dermatitis,70,GSM3427890,0
55636,DSA05050.GSM5788816.Case,GSE193309,GSE193309,1802,495,DSA05050,Skin,19402,Atopic Dermatitis,DOID:3310,Atopic Dermatitis,RNA-Seq,D,DOID:3310,DOID:3310,atopic dermatitis,70,GSM5788816,0
1292,DSA00176.GSM6499445.Case,GSE211700,GSE211700,1715,465,DSA00176,nan,19402,Systemic Lupus Erythematosus,DOID:9074,Systemic Lupus Erythematosus,RNA-Seq,D,DOID:9074,DOID:9074,systemic lupus erythematosus,133,GSM6499445,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44207,DSA03831.GSM4694684.Case,GSE155067,GSE155067,935,259,DSA03831,nan,19402,Schizophrenia,DOID:5419,Schizophrenia,RNA-Seq,D,DOID:5419,DOID:5419,schizophrenia,105,GSM4694684,0
13986,DSA01344.GSM4595391.Control,GSE152004,GSE152004,1931,534,DSA01344,Nasal airway epithelium,19402,Control,Control,Asthma,RNA-Seq,D,Control,Control,Control,0,GSM4595391,0
57203,DSA05072.GSM5978892.Case,GSE193677,GSE193677,1154,313,DSA05072,Bowel,19402,Ulcerative Colitis,DOID:8577,Ulcerative Colitis,RNA-Seq,D,DOID:8577,DOID:8577,ulcerative colitis,123,GSM5978892,0
87883,DSA07529.GSM2371147.Case,GSE89408,GSE89408,1709,463,DSA07529,Synovium,19402,Arthritis,DOID:848,Arthritis,RNA-Seq,D,DOID:848,DOID:848,arthritis,20,GSM2371147,0


# region 3. Load the pre-trained scGPT model

TransformerModel(
  (encoder): GeneEncoder(
    (embedding): Embedding(60697, 512, padding_idx=60694)
    (enc_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  )
  (value_encoder): ContinuousValueEncoder(
    (dropout): Dropout(p=0.5, inplace=False)
    (linear1): Linear(in_features=1, out_features=512, bias=True)
    (activation): ReLU()
    (linear2): Linear(in_features=512, out_features=512, bias=True)
    (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  )
  (transformer_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-11): 12 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
        )
        (linear1): Linear(in_features=512, out_features=512, bias=True)
        (dropout): Dropout(p=0.5, inplace=False)
        (linear2): Linear(in_features=512, out_features=512, bias=True)
        (norm1): LayerNorm((512,), eps=1e-05, el

  2%|▉                                        | 83/3862 [00:19<43:06,  1.46it/s]

In [11]:
from tqdm import tqdm

In [ ]:
celltypes_labels

array(['Control', 'Control', 'Control', ..., 'Schizophrenia',
       'Schizophrenia', 'Schizophrenia'], dtype='<U57')

In [ ]:
adata_query.obs.columns

Index(['ids', 'dataset', 'dataset_id', 'batch', 'batch_id', 'dsaid', 'tissue',
       'n_genes', 'disease', 'celltype', 'disease_study', 'library',
       'doid_study', 'doid_id', 'do_id', 'doid_disease'],
      dtype='object')

In [ ]:

model_path = "/aloy/home/ddalton/projects/scGPT_playground/outputs/run-25-09-17-01/model_1.pt"

model = torch.load(model_path, map_location="cpu")  # try to read container




# load adata
adata_path = "/aloy/home/ddalton/projects/scGPT_playground/outputs/run-25-09-17-01/adata_test_1.h5ad"
import scanpy as sc
adata = sc.read_h5ad(adata_path)